# 02 — Ranking Metrics: Precision@K, Recall@K, MRR, MAP, NDCG

**Overview:** implement every ranking metric from scratch on a tiny, hand-verifiable example,
the kind used to evaluate search and recommendation systems.

**Required libraries:** `numpy`.

Connects to: `01_classification_metrics.ipynb` (threshold-based metrics). Leads to:
`03_cross_validation.ipynb`.

### Abbreviations used in this notebook

| Term | Full form | What it is |
|---|---|---|
| **MRR** | Mean Reciprocal Rank | Average over queries of 1/rank of the first relevant result. |
| **MAP** | Mean Average Precision | Mean over queries of Average Precision (ranking metric). |
| **NDCG** | Normalized Discounted Cumulative Gain | DCG divided by IDCG, so scores fall in 0-1. |
| **RR** | Reciprocal Rank | 1 / rank of the first relevant result for one query. |
| **AP** | Average Precision | Mean of precision@k at each rank where a relevant item appears. |
| **DCG** | Discounted Cumulative Gain | Sum of relevance scores discounted by log of rank position. |
| **IDCG** | Ideal DCG | DCG of the best possible ordering; used to normalise. |

## 0. A tiny, hand-verifiable ranked-results example

In [1]:
import numpy as np

# A single query returned 10 ranked results. 1 = relevant, 0 = not relevant (binary relevance).
ranked_relevance = np.array([1, 0, 1, 1, 0, 0, 1, 0, 0, 0])
print("ranked results (position 1 to 10):", ranked_relevance)
print("total relevant items in this list:", ranked_relevance.sum())


ranked results (position 1 to 10): [1 0 1 1 0 0 1 0 0 0]
total relevant items in this list: 4


## 1. Precision@K and Recall@K

**Equations:** `Precision@K = (# relevant in top K) / K`, `Recall@K = (# relevant in top K) / (total # relevant)`

In [2]:
def precision_at_k(relevance, k):
    return relevance[:k].sum() / k

def recall_at_k(relevance, k, total_relevant=None):
    total_relevant = total_relevant if total_relevant is not None else relevance.sum()
    return relevance[:k].sum() / total_relevant

for k in [1, 3, 5, 10]:
    p = precision_at_k(ranked_relevance, k)
    r = recall_at_k(ranked_relevance, k)
    print(f"K={k:2d}  Precision@K={p:.3f}  Recall@K={r:.3f}")

# Hand-check K=3: top 3 = [1,0,1] -> 2 relevant out of 3 -> Precision@3 = 2/3
assert np.isclose(precision_at_k(ranked_relevance, 3), 2 / 3)
# Recall@3: 2 relevant found out of 4 total relevant -> 2/4 = 0.5
assert np.isclose(recall_at_k(ranked_relevance, 3), 0.5)


K= 1  Precision@K=1.000  Recall@K=0.250
K= 3  Precision@K=0.667  Recall@K=0.500
K= 5  Precision@K=0.600  Recall@K=0.750
K=10  Precision@K=0.400  Recall@K=1.000


## 2. MRR (Mean Reciprocal Rank)

**Equation:** `RR = 1 / (rank of the FIRST relevant result)`; MRR averages this across queries.

💡 **Intuition:** MRR only cares about how quickly you hit the *first* relevant result — perfect
for "did we get at least one good answer near the top" (e.g. a chatbot's first suggestion).

In [3]:
def reciprocal_rank(relevance):
    relevant_positions = np.where(relevance == 1)[0]
    if len(relevant_positions) == 0:
        return 0.0
    first_relevant_rank = relevant_positions[0] + 1   # rank is 1-indexed
    return 1.0 / first_relevant_rank

rr_example = reciprocal_rank(ranked_relevance)
print("first relevant result is at rank:", np.where(ranked_relevance == 1)[0][0] + 1)
print("reciprocal rank:", rr_example)
assert np.isclose(rr_example, 1.0)   # first item (rank 1) IS relevant here

# MRR across multiple queries
queries_relevance = [
    np.array([1, 0, 0]),      # first relevant at rank 1 -> RR=1.0
    np.array([0, 1, 0]),      # first relevant at rank 2 -> RR=0.5
    np.array([0, 0, 0]),      # no relevant result at all -> RR=0.0
]
mrr = np.mean([reciprocal_rank(r) for r in queries_relevance])
print("\nper-query RR:", [reciprocal_rank(r) for r in queries_relevance])
print("MRR:", mrr)
assert np.isclose(mrr, (1.0 + 0.5 + 0.0) / 3)


first relevant result is at rank: 1
reciprocal rank: 1.0

per-query RR: [np.float64(1.0), np.float64(0.5), 0.0]
MRR: 0.5


## 3. MAP (Mean Average Precision)

**Equation:** `AP = average of Precision@k, evaluated at every rank k where a relevant item appears`; MAP averages AP across queries.

In [4]:
def average_precision(relevance):
    relevant_positions = np.where(relevance == 1)[0]
    if len(relevant_positions) == 0:
        return 0.0
    precisions_at_hits = [precision_at_k(relevance, pos + 1) for pos in relevant_positions]
    return np.mean(precisions_at_hits)

ap_example = average_precision(ranked_relevance)
print("relevant positions (0-indexed):", np.where(ranked_relevance == 1)[0])
print("Precision@K evaluated at each relevant hit:",
      [round(precision_at_k(ranked_relevance, pos + 1), 3) for pos in np.where(ranked_relevance == 1)[0]])
print("Average Precision:", round(ap_example, 4))

# Hand-check: relevant at ranks 1, 3, 4, 7 -> P@1=1/1, P@3=2/3, P@4=3/4, P@7=4/7
expected_ap = np.mean([1/1, 2/3, 3/4, 4/7])
assert np.isclose(ap_example, expected_ap)

map_score = np.mean([average_precision(r) for r in queries_relevance])
print("\nMAP across the 3 queries from the MRR example:", round(map_score, 4))


relevant positions (0-indexed): [0 2 3 6]
Precision@K evaluated at each relevant hit: [np.float64(1.0), np.float64(0.667), np.float64(0.75), np.float64(0.571)]
Average Precision: 0.747

MAP across the 3 queries from the MRR example: 0.5


## 4. NDCG (Normalized Discounted Cumulative Gain)

**Equations:** `DCG@K = Σ relevance_i / log2(i+1)` (rewards relevant items more at higher ranks);
`NDCG@K = DCG@K / IDCG@K` (IDCG = DCG of the *ideal*, perfectly-sorted ranking).

💡 **Intuition:** NDCG handles *graded* relevance (not just 0/1) and explicitly discounts items
further down the list.

In [5]:
def dcg_at_k(relevance, k):
    relevance = np.asarray(relevance[:k], dtype=float)
    positions = np.arange(1, len(relevance) + 1)
    return np.sum(relevance / np.log2(positions + 1))

def ndcg_at_k(relevance, k):
    actual_dcg = dcg_at_k(relevance, k)
    ideal_relevance = np.sort(relevance)[::-1]   # best possible ordering
    ideal_dcg = dcg_at_k(ideal_relevance, k)
    return actual_dcg / ideal_dcg if ideal_dcg > 0 else 0.0

# Graded relevance example: scores 0-3 instead of just 0/1.
graded_relevance = np.array([3, 1, 0, 2, 0])
for k in [3, 5]:
    ndcg = ndcg_at_k(graded_relevance, k)
    print(f"NDCG@{k}: {ndcg:.4f}")

# A perfectly-ordered ranking should have NDCG of exactly 1.0 at any K.
perfectly_ordered = np.sort(graded_relevance)[::-1]
assert np.isclose(ndcg_at_k(perfectly_ordered, 5), 1.0)
print("\na perfectly-ordered ranking (best items first) gives NDCG=1.0, as expected.")


NDCG@3: 0.7625
NDCG@5: 0.9434

a perfectly-ordered ranking (best items first) gives NDCG=1.0, as expected.


## 5. Connection to what's next

These ranking metrics are exactly what evaluates a recommendation system or search engine's
output quality — the same evaluation discipline (train/val/test, no leakage) from
`11_generalization` and `03_cross_validation.ipynb` applies when tuning a ranking model too.

## 6. Interview questions
1. Why does MRR only care about the position of the FIRST relevant result, and when is that the right thing to optimize?
2. What's the difference between Precision@K and MAP?
3. Why does NDCG need an "ideal" ranking to normalize against?
4. When would you use graded relevance (NDCG) instead of binary relevance (MAP/MRR)?
5. Why does DCG discount relevance by `log2(rank+1)` rather than linearly by rank?

## 7. Exercises
1. Compute Precision@K and Recall@K for K=1 through 10 on a list with 6 relevant items out of 10, and plot both curves.
2. Build a ranked list where MRR is high but MAP is low, and explain why that combination occurs.
3. Compare NDCG@5 for two rankings with the same items but different orderings of graded relevance.

## 8. Summary

✅ Precision@K/Recall@K, MRR, MAP, and NDCG each answer a different question about ranking
quality — MRR for "how fast is the first good result," MAP for "how good is the ranking
overall across all relevant items," NDCG for "how good is the ranking with graded relevance
and position discounting." Next: `03_cross_validation.ipynb`.